<a href="https://colab.research.google.com/github/leilacielok/market_basket_analysis/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algorithms for Massive Data - Market-Basket Analysis

In [1]:
import os
import pandas as pd

from google.colab import userdata
from itertools import combinations
from collections import Counter

In [12]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")


# SOSTITUIRE CON: Insert your Kaggle credentials before running
# os.environ['KAGGLE_USERNAME'] = "xxxx"
# os.environ['KAGGLE_KEY'] = "xxxx"

!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
!unzip -o -q imdb-dataset-of-top-1000-movies-and-tv-shows.zip -d imdb_data

print("Dataset downloaded and extracted.")

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
License(s): CC0-1.0
imdb-dataset-of-top-1000-movies-and-tv-shows.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset downloaded and extracted.


In [13]:
csv_path = os.path.join(
    "imdb_data",
    "imdb_top_1000.csv"
)

df = pd.read_csv(csv_path)

In [33]:
# Minimum support threshold
MIN_SUPPORT_RATIO = 0.005
MIN_SUPPORT_COUNT = int(len(df) * MIN_SUPPORT_RATIO)

print("Minimum support count:", MIN_SUPPORT_COUNT)

Minimum support count: 5


In [28]:
# Basket construction
STAR_COLS = ["Star1", "Star2", "Star3", "Star4"]

baskets = []

for _, row in df.iterrows():
    basket = []

    for col in STAR_COLS:
        actor = row[col]
        basket.append(actor)

    baskets.append(set(basket))

print("Number of baskets:", len(baskets))
print("Example basket:", baskets[0])

Number of baskets: 1000
Example basket: {'Morgan Freeman', 'Bob Gunton', 'Tim Robbins', 'William Sadler'}


## A-Priori Algorithm

In [34]:
def apriori_pairs(baskets, support_threshold):

    # FIRST PASS

    # Translate item names into integers
    names2int = {}
    int2names = {}
    item_id = 0

    for basket in baskets:
        for item in basket:
            if item not in names2int:
                names2int[item] = item_id
                int2names[item_id] = item
                item_id += 1

    # Use integers as indexes in the counts array
    counts = [0] * len(names2int)

    for basket in baskets:
        for item in basket:
            item_integer = names2int[item]
            counts[item_integer] += 1

    # IN-BETWEEN PASSES

    # Find frequent singletons
    # Create frequent-items table
    frequent_items_table = [0] * len(counts)

    new_numbering = 1

    for item_integer, count in enumerate(counts):
        if count >= support_threshold:
            frequent_items_table[item_integer] = new_numbering
            new_numbering += 1

    # SECOND PASS

    # Count all pairs of frequent items
    pair_counts = {}

    for basket in baskets:

        # a. Find frequent items in the basket
        frequent_items_in_basket = []

        for item in basket:
            old_integer = names2int[item]
            new_integer = frequent_items_table[old_integer]

            if new_integer != 0:
                frequent_items_in_basket.append(new_integer)

        # b. Generate all pairs of frequent items
        for i in range(len(frequent_items_in_basket)):
            for j in range(i + 1, len(frequent_items_in_basket)):

                item1 = frequent_items_in_basket[i]
                item2 = frequent_items_in_basket[j]
                pair = tuple(sorted((item1, item2)))

                # c. Add one to the pair count
                if pair not in pair_counts:
                    pair_counts[pair] = 0

                pair_counts[pair] += 1

    # Keep only frequent pairs
    frequent_pairs = {}

    for pair, count in pair_counts.items():
      if count >= support_threshold:
        actor1 = int2names[pair[0] - 1]
        actor2 = int2names[pair[1] - 1]

        frequent_pairs[(actor1, actor2)] = count

    return (
    names2int,
    counts,
    frequent_items_table,
    frequent_pairs
)

In [36]:
names2int, counts, frequent_items_table, frequent_pairs = apriori_pairs(
    baskets,
    MIN_SUPPORT_COUNT
)

print("Distinct actors:", len(names2int))
print("Frequent actors:", sum(x != 0 for x in frequent_items_table))
print("Frequent pairs:", len(frequent_pairs))

for pair, count in sorted(frequent_pairs.items(),
                          key=lambda x: x[1],
                          reverse=True):
    print(pair, "->", count)

Distinct actors: 2709
Frequent actors: 79
Frequent pairs: 3
('Ray Liotta', 'Carrie Fisher') -> 6
('Ray Liotta', 'Billy Dee Williams') -> 5
('Billy Dee Williams', 'Carrie Fisher') -> 5


In [25]:
frequent_itemsets = {}

# 1-itemsets
item_counts = Counter()

for basket in baskets:
    for item in basket:
        item_counts[frozenset([item])] += 1

current_frequent = {
    itemset: count
    for itemset, count in item_counts.items()
    if count >= MIN_SUPPORT_COUNT
}

frequent_itemsets[1] = current_frequent

print("Frequent 1-itemsets:", len(current_frequent))


# k-itemsets
k = 2

while current_frequent:
    candidate_counts = Counter()

    frequent_items_previous = list(current_frequent.keys())

    candidates = set()

    for itemset1 in frequent_items_previous:
        for itemset2 in frequent_items_previous:
            candidate = itemset1.union(itemset2)

            if len(candidate) == k:
                candidates.add(candidate)

    for basket in baskets:
        for candidate in candidates:
            if candidate.issubset(basket):
                candidate_counts[candidate] += 1

    current_frequent = {
        itemset: count
        for itemset, count in candidate_counts.items()
        if count >= MIN_SUPPORT_COUNT
    }

    if current_frequent:
        frequent_itemsets[k] = current_frequent
        print(f"Frequent {k}-itemsets:", len(current_frequent))

    k += 1

Frequent 1-itemsets: 79
Frequent 2-itemsets: 3
Frequent 3-itemsets: 1


In [26]:
# Display frequent itemsets

for size, itemsets in frequent_itemsets.items():
    print("\n" + "=" * 50)
    print(f"Frequent itemsets of size {size}")
    print("=" * 50)

    sorted_itemsets = sorted(
        itemsets.items(),
        key=lambda x: x[1],
        reverse=True
    )

    for itemset, count in sorted_itemsets[:20]:
        print(set(itemset), "-> support count:", count)


Frequent itemsets of size 1
{'Robert De Niro'} -> support count: 17
{'Tom Hanks'} -> support count: 14
{'Al Pacino'} -> support count: 13
{'Brad Pitt'} -> support count: 12
{'Clint Eastwood'} -> support count: 12
{'Christian Bale'} -> support count: 11
{'Leonardo DiCaprio'} -> support count: 11
{'Matt Damon'} -> support count: 11
{'James Stewart'} -> support count: 10
{'Michael Caine'} -> support count: 9
{'Scarlett Johansson'} -> support count: 9
{'Humphrey Bogart'} -> support count: 9
{'Ethan Hawke'} -> support count: 9
{'Johnny Depp'} -> support count: 9
{'Denzel Washington'} -> support count: 9
{'Harrison Ford'} -> support count: 8
{'Aamir Khan'} -> support count: 8
{'Morgan Freeman'} -> support count: 7
{'Ian McKellen'} -> support count: 7
{'Bruce Willis'} -> support count: 7

Frequent itemsets of size 2
{'Daniel Radcliffe', 'Rupert Grint'} -> support count: 6
{'Daniel Radcliffe', 'Emma Watson'} -> support count: 5
{'Emma Watson', 'Rupert Grint'} -> support count: 5

Frequent ite

In [22]:
print("Frequent triples")

for itemset, count in sorted(frequent_itemsets.get(3, {}).items(),
                             key=lambda x: x[1],
                             reverse=True):
    print(set(itemset), "-> support count:", count)

Frequent triples
